# Youtu-LLM-2B approximate attention local GPU test

This notebook loads the implementation from `mla_santapp.py`. Prefill remains dense; one-token decoding compares dense attention with cluster-ranked KV reads.

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

prompt = "Explain why the sky is blue."
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "tencent/Youtu-LLM-2B"

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)


Loading weights:   0%|          | 0/386 [00:00<?, ?it/s]

In [ ]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False)
input_ids = tokenizer(text, return_tensors="pt")
ids = input_ids.input_ids.to(device)
print(ids.shape)
out = model(ids, use_cache=True)
print("ids length", len(ids))
pkv = out.past_key_values
print("keys", pkv.layers[0].keys.shape)

print("values", pkv.layers[0].values.shape)
pkv.layers[0].keys[..., 64:] == pkv.layers[0].values

torch.Size([1, 14])
ids length 1
keys torch.Size([1, 16, 14, 192])
values torch.Size([1, 16, 14, 128])


RuntimeError: The size of tensor a (129) must match the size of tensor b (128) at non-singleton dimension 3